# Exercise for Graph IV: Conditional Graph Pipeline

### Your Task:
Build a graph with **2 conditional edges** performing two consecutive mathematical operations based on condition operators:
- **Stage 1**: Computes `finalNumber` based on `operation` (`"+"` or `"-"`) using `number1` and `number2`.
- **Stage 2**: Computes `finalNumber2` based on `operation2` (`"+"` or `"-"`) using `number3` and `number4`.

**Input State:**
```python
initial_state = {
    "number1": 10,
    "operation": "-",
    "number2": 5,
    "number3": 7,
    "number4": 2,
    "operation2": "+",
    "finalNumber": 0,
    "finalNumber2": 0
}
```

**Expected Output State:**
- `finalNumber = 5` (since `10 - 5 = 5`)
- `finalNumber2 = 9` (since `7 + 2 = 9`)

In [1]:
# Install dependencies if needed:
# !pip install langgraph langchain-core

## 1. Define the State

We define `AgentState` with TypedDict matching all the fields.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict, total=False):
    number1: int
    operation: str
    number2: int
    number3: int
    number4: int
    operation2: str
    finalNumber: int
    finalNumber2: int

## 2. Define Stage 1 and Stage 2 Nodes and Routing Conditions

In [3]:
# --- Stage 1 Nodes (Operates on number1 and number2) ---
def add_node1(state: AgentState):
    return {"finalNumber": state["number1"] + state["number2"]}

def subtract_node1(state: AgentState):
    return {"finalNumber": state["number1"] - state["number2"]}

# Conditional Router for Stage 1
def route_stage1(state: AgentState):
    if state["operation"] == "+":
        return "add_node1"
    else:
        return "subtract_node1"

# --- Stage 2 Nodes (Operates on number3 and number4) ---
def add_node2(state: AgentState):
    return {"finalNumber2": state["number3"] + state["number4"]}

def subtract_node2(state: AgentState):
    return {"finalNumber2": state["number3"] - state["number4"]}

# Conditional Router for Stage 2
def route_stage2(state: AgentState):
    if state["operation2"] == "+":
        return "add_node2"
    else:
        return "subtract_node2"

## 3. Construct the Graph with 2 Conditional Edges

- **Conditional Edge 1:** From `START`, routes conditionally to `add_node1` or `subtract_node1` based on `operation`.
- **Conditional Edge 2:** From stage 1 nodes, routes conditionally to `add_node2` or `subtract_node2` based on `operation2`.

In [4]:
workflow = StateGraph(AgentState)

# Add all computation nodes
workflow.add_node("add_node1", add_node1)
workflow.add_node("subtract_node1", subtract_node1)
workflow.add_node("add_node2", add_node2)
workflow.add_node("subtract_node2", subtract_node2)

# 1st Conditional Edge: Routing from START to Stage 1 node
workflow.add_conditional_edges(
    START,
    route_stage1,
    {
        "add_node1": "add_node1",
        "subtract_node1": "subtract_node1"
    }
)

# 2nd Conditional Edge: Routing from Stage 1 nodes to Stage 2 node
workflow.add_conditional_edges(
    "add_node1",
    route_stage2,
    {
        "add_node2": "add_node2",
        "subtract_node2": "subtract_node2"
    }
)
workflow.add_conditional_edges(
    "subtract_node1",
    route_stage2,
    {
        "add_node2": "add_node2",
        "subtract_node2": "subtract_node2"
    }
)

# Connect Stage 2 nodes to END
workflow.add_edge("add_node2", END)
workflow.add_edge("subtract_node2", END)

app = workflow.compile()

## 4. Run with Exercise Input

In [5]:
initial_state = {
    "number1": 10,
    "operation": "-",
    "number2": 5,
    "number3": 7,
    "number4": 2,
    "operation2": "+",
    "finalNumber": 0,
    "finalNumber2": 0
}

result = app.invoke(initial_state)

print("Input State:")
print(initial_state)
print("\nFinal Result State:")
print(result)
print("\nResults:")
print(f"finalNumber ({result['number1']} {result['operation']} {result['number2']}) = {result['finalNumber']}")
print(f"finalNumber2 ({result['number3']} {result['operation2']} {result['number4']}) = {result['finalNumber2']}")

Input State:
{'number1': 10, 'operation': '-', 'number2': 5, 'number3': 7, 'number4': 2, 'operation2': '+', 'finalNumber': 0, 'finalNumber2': 0}

Final Result State:
{'number1': 10, 'operation': '-', 'number2': 5, 'number3': 7, 'number4': 2, 'operation2': '+', 'finalNumber': 5, 'finalNumber2': 9}

Results:
finalNumber (10 - 5) = 5
finalNumber2 (7 + 2) = 9
